# RAG

RAG หรือ Retrieval-Augmented Generation คือ เทคนิคการเพิ่มความสามารถให้โมเดลภาษาขนาดใหญ่ (LLM) โดยการดึงข้อมูลจากแหล่งข้อมูลภายนอกมาใช้ประกอบการสร้างคำตอบ แทนที่จะพึ่งพาเพียงความรู้ที่โมเดลมีอยู่เดิมจากการเทรน ซึ่งช่วยลดการ Hallucination และทำให้สามารถตอบคำถามเกี่ยวกับข้อมูลใหม่ๆ หรือข้อมูลเฉพาะชุดนั้นๆ ได้อย่างถูกต้องมากขึ้น

โดยจะมีขั้นตอนการทำงานคร่างๆ 2 ส่วน คือ

#### 1. Indexing 

ขั้นตอนนี้ คือ การทำสารบัญเพื่อให้ AI ค้นหาข้อมูลได้รวดเร็ว ซึ่งเริ่มต้นตั้งแต่

* Load Data: อ่านข้อมูลจากไฟล์
* Chunking: หากข้อมูลยาวเกินไป จะต้องแบ่งเป็นส่วนย่อยๆ
* Embedding: นำข้อความ (ชื่อสินค้า + รายละเอียด) ไปผ่านโมเดล Embedding เพื่อแปลงเป็นเวกเตอร์
* Vector Database: จัดเก็บเวกเตอร์เหล่านั้นไว้ในฐานข้อมูลอย่าง Qdrant

#### 2. Generating 

ขั้นตอนนี้ จะเกิดขึ้นเมื่อมี **คำถาม** จากผู้ใช้เข้ามา โดยเริ่มตั้งแต่

* Embedding: ระบบจะแปลงคำถามของผู้ใช้เป็น Vector
* Retrieving: นำเวกเตอร์ที่ได้ไปค้นหาสินค้าที่ใกล้เคียงที่สุด (Similarity Search) โดยใช้ cosine similarity
* Generating: ส่งข้อมูลทั้งหมด (คำถาม + รายการสินค้าที่ค้นเจอ) ไปให้โมเดลสร้างคำตอบที่อ่านง่ายและเป็นธรรมชาติออกมา 


![RAG Pipeline](https://docs.nvidia.com/nemo-framework/user-guide/24.12/_images/rag_pipeline.png "RAG Pipeline")

credit: https://docs.nvidia.com/nemo-framework/user-guide/24.12/rag/ragoverview.html

In [174]:
# https://python-client.qdrant.tech/

# Task: Product Recommendation Agent

ในแบบฝึกหัดนี้ เราจะสร้างระบบแนะนำสินค้าจาก Shopee โดยใช้ RAG

**เป้าหมาย:** เมื่อผู้ใช้พิมพ์ความต้องการเข้ามา (เช่น "อยากได้รองเท้าวิ่งที่ระบายอากาศดี") ระบบจะไปเลือกสินค้าที่ "ความหมายใกล้เคียง" กับความต้องการมากที่สุดมาแนะนำ

## Step 1: Indexing

In [23]:
import pandas as pd

products = pd.read_csv('./Examples/shopee-products.csv')

products.tail()

,url,id,title,sold,rating,reviews,initial_price,final_price,currency,stock,...,category_id,flash_sale,flash_sale_time,product_variation,gmv_cal,category_url,vouchers,is_available,seller_id,product_ratings
81,https://shopee.co.th/%E0%B8%8A%E0%B8%B8%E0%B8%...,13333008713,ชุดเซรั่ม 4 สูตร แก้ปัญหาผิวได้อย่างตรงจุด ตอบ...,0,4.9,0,2500.0,599.0,THB,NaN,...,100630,False,NaN,"[{""name"":""model"",""value"":null}]",0,https://shopee.co.th/%E0%B8%97%E0%B8%A3%E0%B8%...,"[{""claimable"":false,""currency"":""THB"",""discount...",NaN,NaN,NaN
82,https://shopee.co.th/Campingmoon-%E0%B9%82%E0%...,5350882392,Campingmoon โต๊ะพับเหล็กหล่อ ตะแกรงเหล็กมีความ...,0,4.9,0,1499.0,1499.0,THB,NaN,...,100637,False,NaN,"[{""name"":""model"",""value"":""233-3T กากี""}]",0,https://shopee.co.th/%E0%B9%80%E0%B8%95%E0%B9%...,"[{""claimable"":false,""currency"":""THB"",""discount...",NaN,NaN,NaN
83,https://shopee.co.th/Xiaomi-Redmi-Note-4-6-7-8...,22043883259,Xiaomi Redmi Note 4 6 7 8 9 9s 10 11 12 Case Fila,0,0.0,0,119.0,119.0,THB,NaN,...,100013,False,NaN,"[{""name"":""model"",""value"":""Redmi Note 12 (5G),แ...",0,https://shopee.co.th/%E0%B8%AD%E0%B8%B7%E0%B9%...,NaN,NaN,NaN,NaN
84,https://shopee.co.th/PARFUMS-de-MARLY-DELINA-L...,15948844862,PARFUMS de MARLY DELINA La Rosee 1ml 2ml,0,5.0,0,180.0,180.0,THB,NaN,...,100630,False,NaN,"[{""name"":""model"",""value"":""1ml-180""}]",0,https://shopee.co.th/%E0%B8%99%E0%B9%89%E0%B8%...,NaN,NaN,NaN,NaN
85,https://shopee.co.th/%E0%B8%99%E0%B9%89%E0%B8%...,20892981651,น้ำยาล้างเบรก Motul Brake Clean P2,0,5.0,0,297.0,297.0,THB,NaN,...,100640,False,NaN,"[{""name"":""model"",""value"":null}]",0,https://shopee.co.th/%E0%B8%9C%E0%B8%A5%E0%B8%...,"[{""claimable"":false,""currency"":""THB"",""discount...",NaN,NaN,NaN


In [187]:
products.shape

(86, 37)

In [5]:
# !uv pip install qdrant-client
# !uv pip install fastembed

In [8]:
# !uv pip install ollama

### Create vectors

In [60]:
from ollama import Client

ollama = Client(host='http://localhost:11434')
# model = "scb10x/typhoon2.5-qwen3-4b:latest"
model = "qwen3-embedding:4b"

In [62]:
for m in ollama.list().models:
    print(m.model)

qwen3-embedding:4b
hf.co/mradermacher/typhoon2.5-qwen3-4b-gguf:Q4_K_M
qwen3-vl:4b-instruct
scb10x/typhoon2.5-qwen3-4b:latest
hf.co/typhoon-ai/typhoon2.5-qwen3-4b-gguf:Q4_K_M
qwen3-vl:4b
scb10x/typhoon2.1-gemma3-4b:latest


#### เลือก Embedding Models ยังไง?

เบื้องต้น สามารถดูประสิทธิภาพโมเดลได้ที่ [MTEB Leaderboard](https://huggingface.co/spaces/mteb/leaderboard)

แต่ แต่ แต่ ต้องทดสอบเองอีกทีว่าโมเดลที่เลือกมามันใช้ได้ตรงกับที่เราต้องการแค่ไหน?

อ่านเพ่ิมเติมได้ที่: https://qdrant.tech/articles/how-to-choose-an-embedding-model/

#### Embedding Models ต่างจาก Based Model ยังไง?

Embedding Models โดยทั่วไปจะเป็นการเทรน Based Model ให้สามารถผลิต embedding ที่มีคุณภาพเพิ่มเติม เช่น
* (InfoNCE) Contrastive Loss
* Matryoshka Representation Learning (MRL)
* ...

เช่น 
* https://arxiv.org/abs/2511.07025
* https://arxiv.org/abs/2503.07891
* https://arxiv.org/abs/2506.05176

In [176]:
from fastembed import TextEmbedding, LateInteractionTextEmbedding, SparseTextEmbedding 
from tqdm import tqdm
import json

# dense_embedding_model = TextEmbedding("sentence-transformers/all-MiniLM-L6-v2")
# late_interaction_embedding_model = LateInteractionTextEmbedding("colbert-ir/colbertv2.0")

texts = []
for idx, row in tqdm(products.iterrows(), total=len(products)):
    # print(row)
    text = f""
    text += f"ชื่อ: {row['title']}\n"
    text += f"คะแนน: {row['rating']}\n"
    text += f"ราคา: {row['final_price']}\n"
    text += f"ประเภทสินค้น: {' / '.join(json.loads(row['breadcrumb']))}\n"
    text += f"รายละเอียด: \n{row['Product Description']}\n"
    for var in json.loads(row["variations"]):
        if 'name' not in var:
            continue
            
        text += f"\t{var['name']}: {', '.join(var['variations'])}\n"
    texts.append(text)

100%|████████████████████████████████████████████████████████████████████████████████████| 86/86 [00:00<00:00, 25242.14it/s]


In [177]:
print(texts[0])

ชื่อ: เคสหนัง กันกระแทก ลายดอกคามิเลีย แฟชั่น สําหรับ Apple Airpods 1 2 3rd Pro 2 Airpods Pro 2
คะแนน: 4.8
ราคา: 117.0
ประเภทสินค้น: มือถือและอุปกรณ์เสริม / อุปกรณ์เสริม / เคสมือถือและแท็บเล็ต / อื่นๆ
รายละเอียด: 
-------🌸ยินดีต้อนรับสู่ร้านค้า myiwatch🌸-------
 
 🔔เราเป็นร้านขายอุปกรณ์ Apple ที่น่าสนใจหากคุณชอบผลิตภัณฑ์ของเรา
 เราจะมีสินค้าลดราคาในอนาคตโปรดติดตามเราเราจะมีสินค้าลดราคาเพิ่มเติมในอนาคต🎁🎁🎁🎁🎁
 💟บริการลูกค้าออนไลน์เวลา 10: 00-23: 00 น
 💟สินค้าอยู่ในสต็อกและสามารถจัดส่งได้ภายใน 48 ชั่วโมง
 💟หากคุณมีคำถามใด ๆ เกี่ยวกับผลิตภัณฑ์โปรดปรึกษาเรา
 
 ✂✂✂✂✂✂✂✂✂✂✂✂✂✂✂✂✂✂✂✂✂✂
 ใหม่เอี่ยม -📷แฟชั่นเคสหนังรูปสี่เหลี่ยมขนมเปียกปูนแฟชั่น
 ❤รายละเอียดสินค้า:
 ✳รุ่นที่ใช้งานได้: สำหรับ Airpods 1/2, Airpods pro, Airpods 3, Aripods pro2
 ✳สไตล์: ฝาหลัง
 ✳วัสดุ: หนัง
 ✳องค์ประกอบยอดนิยม: แฟชั่น
 ✳กระบวนการ: การฉีด / การฉีด
 
 💋วิธีใช้:
 ทำความสะอาด Airpods Airpods แบบสะอาด 1 ตัว
 ใช้ซิลิโคนป้องกัน 2』 เชื่อมต่อฝาครอบซิลิโคนอย่างระมัดระวัง
 3』 บีบอากาศออกช้าๆ
 เสื้อผ้าที่เหมาะสมไม่หลุดร่วง 4』 เสื

In [180]:
bm25Model = SparseTextEmbedding("Qdrant/bm25")
sparseEmbeddings = list(bm25Model.embed(doc for doc in texts))

In [186]:
np.min(sparseEmbeddings[0].indices)

np.int64(6048359)

In [188]:
from tqdm import tqdm

embeddings = []
for idx, doc in tqdm(enumerate(texts), total=len(texts)):
    response = ollama.embed(model = model, input = doc)
    embeddings.append(response.embeddings)
    

100%|███████████████████████████████████████████████████████████████████████████████████████| 86/86 [06:06<00:00,  4.27s/it]


In [189]:
import numpy as np
print("Embedding Size:", np.array(response.embeddings).shape)

Embedding Size: (1, 2560)


### Create Vector DB

In [190]:
from qdrant_client import models

In [191]:
from qdrant_client.models import Distance, VectorParams
from qdrant_client import QdrantClient
qclient = QdrantClient(":memory:")
# qclient = QdrantClient("http://localhost:6333")

COLLECTION_NAME = "products"
qclient.create_collection(
    collection_name = COLLECTION_NAME,
    vectors_config = {
        "qwen3-embedding:4b": VectorParams(size = 2560, distance = Distance.COSINE),
    },
    sparse_vectors_config={
        "bm25": models.SparseVectorParams(modifier=models.Modifier.IDF)
    }
)

True

In [192]:
productInfos = products.to_dict(orient='records')

In [193]:
from qdrant_client import models

for idx, (doc, info, emb, spEmb) in tqdm(enumerate(zip(texts, productInfos, embeddings, sparseEmbeddings)), total=len(texts)):
    qclient.upsert(
        collection_name = COLLECTION_NAME,
        points=[models.PointStruct(
            id = idx, 
            vector = {
                "qwen3-embedding:4b": emb,
                "bm25": spEmb.as_object(),
            }, 
            payload = {"text": doc, **info})],
    )

100%|█████████████████████████████████████████████████████████████████████████████████████| 86/86 [00:00<00:00, 1709.61it/s]


## STEP2: Generating

In [194]:
def get_embedding(text):
    dense_vectors = ollama.embeddings(model = model, prompt = text).embedding

    sparse_vectors = bm25Model.query_embed(text)
    sparse_vectors = list(sparse_vectors)

    return dense_vectors, sparse_vectors

In [195]:
query_text = "อยากได้เคสมือถือใหม่จัง"
dense_vectors, sparse_vectors = get_embedding(query_text)

In [197]:
from qdrant_client import QdrantClient, models

def rag_search(dense_vectors, sparse_vectors):
    search_results = qclient.query_points(
        collection_name = COLLECTION_NAME,
        prefetch=[
            models.Prefetch(
                query=dense_vectors,
                using="qwen3-embedding:4b",
                limit=10,
            ),
            models.Prefetch(
                query=models.SparseVector(**sparse_vectors[0].as_object()),
                using="bm25",
                limit=10,
            ),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=3,
    )
    
    for hit in search_results.points:
        print(f"Score: {hit.score}")
        print(f"Text: {hit.payload['title']}")
        print()

    return search_results

search_results = rag_search(dense_vectors, sparse_vectors)

Score: 1.0
Text: Xiaomi Redmi Note 4 6 7 8 9 9s 10 11 12 Case Fila

Score: 0.6666666666666666
Text: เคสใส สำหรับ iPhone รุ่นใหม่ล่าสุด 🔥 TPU 🔥เคสกันกระแทก รุ่น 15/14 Pro Max 13/12 Pro Max/11 pro/XS Max/XR/X|8/7 Plus#02

Score: 0.5
Text: เคสหนัง กันกระแทก ลายดอกคามิเลีย แฟชั่น สําหรับ Apple Airpods 1 2 3rd Pro 2 Airpods Pro 2



In [198]:
from pydantic import BaseModel, Field

class SelectedProduct(BaseModel):
    index: int = Field(description="The index of the selected product from the provided list")
    product_name: str = Field(description="The name of the chosen product")
    reasoning: str = Field(description="Brief explanation of why this product matches the query")
    confidence_score: float = Field(ge=0, le=1, description="Score between 0 and 1")
    
def rerank_with_ollama(query_text, search_results):
    """
    Sends the query and a list of candidates to Ollama to find the best match.
    """
    context = "\n".join([f"ID {i}: \n{doc.payload['text']}\n\n{'='*50}\n" for i, doc in enumerate(search_results.points)])
    
    prompt = f"""
    You are an expert product recommender. 
    User Query: "{query_text}"
    
    Candidates:
    {context}
    
    Based on the query, which ID is the absolute best match? 
    
    Return a JSON object with these keys:
    - "selected_index": (int) The index of the best match
    - "confidence_score": (float between 0 and 1)
    - "reasoning": (string) Why this fits the user query
    """
    
    response = ollama.chat(
        model="scb10x/typhoon2.5-qwen3-4b:latest", 
        messages=[
            {
              'role': 'user',
              'content': prompt,
            }
        ],
        format=SelectedProduct.model_json_schema(),
    )
    
    o = SelectedProduct.model_validate_json(response.message.content)
    print("ReRanking Result:", o)
    return o

reRankResponse = rerank_with_ollama(query_text, search_results)

ReRanking Result: index=1 product_name='เคสใส สำหรับ iPhone รุ่นใหม่ล่าสุด 🔥 TPU 🔥เคสกันกระแทก รุ่น 15/14 Pro Max 13/12 Pro Max/11 pro/XS Max/XR/X|8/7 Plus#02' reasoning="The user's query is 'อยากได้เคสมือถือใหม่จัง', which translates to 'I want a new phone case'. The query emphasizes wanting a new phone case, indicating a desire for a current, modern, and protective case. Among the candidates, ID 1 offers a new, trendy TPU case designed specifically for the latest iPhone models (including 14, 15, 13, etc.), which aligns perfectly with the user's request for a 'new' phone case. The case is made from durable TPU, includes protective design features (raised edges), and is explicitly labeled for the latest iPhone models, making it highly relevant and timely. In contrast, ID 0 is limited to older Redmi Note models and lacks clarity on being 'new'. ID 2 is for AirPods cases, which are earbuds and not phone cases. Therefore, ID 1 is the absolute best match." confidence_score=0.98


In [199]:
reRankResponse.index

1

In [200]:
context = "\n".join([f"ID {i}: \n{doc.payload['title']}\n\n{'='*50}\n" for i, doc in enumerate(search_results.points)])
print(context)

ID 0: 
Xiaomi Redmi Note 4 6 7 8 9 9s 10 11 12 Case Fila


ID 1: 
เคสใส สำหรับ iPhone รุ่นใหม่ล่าสุด 🔥 TPU 🔥เคสกันกระแทก รุ่น 15/14 Pro Max 13/12 Pro Max/11 pro/XS Max/XR/X|8/7 Plus#02


ID 2: 
เคสหนัง กันกระแทก ลายดอกคามิเลีย แฟชั่น สําหรับ Apple Airpods 1 2 3rd Pro 2 Airpods Pro 2




In [201]:
search_results.points[reRankResponse.index].payload.keys()

dict_keys(['text', 'url', 'id', 'title', 'sold', 'rating', 'reviews', 'initial_price', 'final_price', 'currency', 'stock', 'favorite', 'image', 'video', 'seller_name', 'shop_url', 'breadcrumb', 'Product Specifications', 'Product Description', 'seller_rating', 'seller_products', 'seller_chats_responded_percentage', 'seller_chat_time_reply', 'seller_joined_date', 'seller_followers', 'variations', 'domain', 'brand', 'category_id', 'flash_sale', 'flash_sale_time', 'product_variation', 'gmv_cal', 'category_url', 'vouchers', 'is_available', 'seller_id', 'product_ratings'])

In [203]:
def generate_persona_response(userQuery, searchResults, reRankResponse):
    # # Define the Persona Instructions
    # system_prompt = (
    #     "You are a friendly, enthusiastic sale from Thailand. "
    #     "You speak politely and you must answer in Thai."
    # )

    selectedProduct = searchResults.points[reRankResponse.index].payload

    user_prompt = f"""
    User Query: {userQuery}
    Selected Product: {selectedProduct['title']}
    Reranker Reasoning: {reRankResponse.reasoning}
    Product Details: {selectedProduct['text']}
    Product URL: {selectedProduct['url']}
    
    Persona: Act as a Trendy Thai Influencer. Your tone should be friendly and upbeat. Use common Thai particles like krub/ka or na where appropriate to make it feel authentic.
    
    Task: Recommend the given product based on the given persona. Make sure to send product's url as a suggestion.
    """

    response = ollama.generate(
        # messages=[
        #     {'role': 'system', 'content': system_prompt},
        #     {'role': 'user', 'content': user_prompt},
        # ],
        prompt = user_prompt,
        model='scb10x/typhoon2.5-qwen3-4b:latest',
    )

    return response

response = generate_persona_response(query_text, search_results, reRankResponse)

In [206]:
print(response.response)

โอ้ยยย!! อยากได้เคสมือถือใหม่จัง นี่มันต้องเป็นเคสใส TPU น่ารักๆ เรียบๆ แต่กันกระแทกดีๆ ต้องเป็นแบบนี้เลยน้าาาา 😍✨  

เคสใสสำหรับ iPhone รุ่นใหม่ล่าสุดนี้มาแรงมากเลยยยย!! 🔥 ใช้วัสดุ TPU หนา 2.3 มม. เนียนๆ เบาๆ เดินทางไปไหนก็ไม่กลัวกระแทกหน้าจอหรือกล้องอ่ะ💕 ขอบจอสูงกว่าหน้าจอ 1 มม. ป้องกันได้ดีสุดๆ เลยยยย 🛡️  

แถมเคสนี้ออกแบบโดยนักออกแบบมืออาชีพจากไทยเองน้าาาาาาาาาาาาาาาาาาาาาาาาาาาาาาาา


In [168]:
def sale_bot(query_text):
    print("Generating query embedding...")
    dense_vectors, sparse_vectors = get_embedding(query_text)
    print("Fetching...")
    search_results = rag_search(dense_vectors, sparse_vectors)
    print("Reranking...")
    reRank_response = rerank_with_ollama(query_text, search_results)

    print("Generating final response...")
    response = generate_persona_response(query_text, search_results, reRank_response)
    print()
    print("Response:")
    print(response.response)
    
sale_bot("อยากได้เคสมือถือใหม่จัง")

Generating query embedding...
Fetching...
Score: 1.0
Text: Xiaomi Redmi Note 4 6 7 8 9 9s 10 11 12 Case Fila

Score: 0.6666666666666666
Text: เคสใส สำหรับ iPhone รุ่นใหม่ล่าสุด 🔥 TPU 🔥เคสกันกระแทก รุ่น 15/14 Pro Max 13/12 Pro Max/11 pro/XS Max/XR/X|8/7 Plus#02

Score: 0.5
Text: เคสหนัง กันกระแทก ลายดอกคามิเลีย แฟชั่น สําหรับ Apple Airpods 1 2 3rd Pro 2 Airpods Pro 2

Reranking...
Generating final response...

Response:
เฮ้ยยยย! คุณกำลังจะได้รับเคสโทรศัพท์ที่น่ารักปังปังแบบมืออาชีพมากกว่าเดิมเลยนะาาา 😍  
ฉันเพิ่งเห็นเคสใสใหม่ล่าสุดนี้จากประเทศไทยเลยยยย! 🔥 มันคือเคสใส TPU ดีงามๆ สำหรับ iPhone รุ่นใหม่ล่าสุดเลยนะาาา ทั้ง 15/14 Pro Max, 13/12 Pro Max, 11 Pro, XS Max เลยครับว่าครอบคลุมทุกรุ่นเลยยยย!  

✨ ขอบจอสูงกว่าหน้าจอ 1 มม. ช่วยป้องกันหน้าจอและกล้องเมื่อทำตกได้ดีมากกกก  
✨ วัสดุ TPU หนา 2.3 มม. ทนทาน ไม่ขรุขระ ผิวเรียบเนียนนนน  
✨ มีนักออกแบบมืออาชีพคนไทยออกแบบมาโดยเฉพาะเลยนะาาา ดูน่ารักน่ารักน่ารักเลยยยย 💖  

แถมสินค้าพร้อมส่งจากประเทศไทยในเวลาไม่กี่วันนี้เองด้วยล่ะาาา 🚀  
สีขาวและเทา

In [173]:
sale_bot("หิวจังเลยกินอะไรดี")

Generating query embedding...
Fetching...
Score: 0.5
Text: ถั่วตัด Good Taste Peanut Brittle (ရသာ ကောင်း မြေ ပဲ ယို) ถั่วตัดพม่า ขนมพม่า อร่อย กลมกล่อม ไม่เหนียว

Score: 0.5
Text: [ลูกค้าใหม่ราคา 1 บาท]🍎รองเท้านักเรียนโกลซิตี้ GCรุ่นFC001/Matin T205 ทนชาย หญิง น้ำตาล ขาว ดำ ไซร์31-45(มีบิลเบิกรรให้)

Score: 0.3333333333333333
Text: Valentino women's rockstud heeled flip flops caged sandal summer street footwear size35-40

Reranking...
ReRanking Result: index=-1 product_name='' reasoning="The user query 'หิวจังเลยกินอะไรดี' translates to 'I'm so hungry, what should I eat?' indicating a request for food or snacks. None of the provided candidates are food items. ID 0 is a snack (peanut brittle), which is food-related, while IDs 1 and 2 are footwear products unrelated to food. Given that the query specifically asks for something to eat, and ID 0 is the only candidate that qualifies as a food/snack, it is the best match despite its low score. The absence of any food items in the other candi

# What more you can do?

1. Semantic Chunking
2. Metadata filtering
3. Hierarchical indexing
4. Query rewriting
5. Query decomposition
6. ... and more